In [20]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
import uuid
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel, Field
from typing import Optional, List
import datetime
import time
import os


In [21]:
class PersonDetails(BaseModel):
    first_name: str = Field(default="Unknown", description="First name.")
    last_name: str = Field(default="Unknown", description="Last name.")
    address: str = Field(default="Unknown", description="Full address.")
    phone: str = Field(default="Unknown", description="Phone number.")

class SuspectDescription(BaseModel):
    race: str = Field(default="Unknown", description="Race.")
    gender: str = Field(default="Unknown", description="Gender.")
    clothing: str = Field(default="Unknown", description="Clothing.")
    physical_features: str = Field(default="Unknown", description="Features.")

class VehicleDescription(BaseModel):
    make: str = Field(default="Unknown", description="Make.")
    model: str = Field(default="Unknown", description="Model.")
    color: str = Field(default="Unknown", description="Color.")
    plate_number: str = Field(default="Unknown", description="Plate.")

In [22]:
class IncidentReport(BaseModel):
    reporting_person: PersonDetails = Field(default_factory=PersonDetails)
    incident_type: str = Field(default="Unknown", description="Crime type (Theft, Damage, etc).")
    when_time: str = Field(default="Unknown", description="Date/Time.")
    where_location: str = Field(default="Unknown", description="Location.")
    what_happened: str = Field(default="Unknown", description="Narrative.")
    suspect_info: SuspectDescription = Field(default_factory=SuspectDescription)
    vehicle_info: VehicleDescription = Field(default_factory=VehicleDescription)

In [23]:
@tool(args_schema=IncidentReport)
def submit_incident_report(**kwargs):
    """
    Call this ONLY when the user says 'Yes' to the summary.
    This tool validates completeness before saving.
    """
    try:
      
        report = IncidentReport(**kwargs)

        timestamp = int(time.time())
        filename = f"Incident_Report_{timestamp}.txt"
        filepath = os.path.join(".", filename)
    
        file_content = (
            f"INCIDENT REPORT #{timestamp}\n"
            f"========================================\n"
            f"REPORTING PARTY: {report.reporting_person.first_name} {report.reporting_person.last_name}\n"
            f"ADDRESS:         {report.reporting_person.address}\n"
            f"PHONE:           {report.reporting_person.phone}\n"
            f"----------------------------------------\n"
            f"LOCATION:        {report.where_location}\n"
            f"TIME:            {report.when_time}\n"
            f"----------------------------------------\n"
            f"NARRATIVE:\n{report.what_happened}\n"
            f"----------------------------------------\n"
            f"SUSPECT INFO:    {report.suspect_info.model_dump_json()}\n"
            f"VEHICLE INFO:    {report.vehicle_info.model_dump_json()}\n"
            f"========================================\n"
            f"STATUS: FILED"
        )

        with open(filepath, "w") as f:
            f.write(file_content)
        return f"Report saved to {filename}."

    except ValidationError as e:
        return f"DATA FORMAT ERROR: {str(e)}"
    except Exception as e:
        return f"SYSTEM ERROR: {str(e)}"

@tool
def calculate_date(relative_time: str) -> str:
    """
    Calculates the exact date based on a relative description (e.g., '2 days ago', 'last Friday').
    Always call this when the user gives a relative time.
    """
    today = datetime.date.today()
    relative_time = relative_time.lower()
    
    try:
        if "yesterday" in relative_time:
            target_date = today - datetime.timedelta(days=1)
        elif "ago" in relative_time and "day" in relative_time:
            # Extract number (e.g., "2 days ago")
            parts = relative_time.split()
            days = 1
            for part in parts:
                if part.isdigit():
                    days = int(part)
                    break
            target_date = today - datetime.timedelta(days=days)
        elif "week" in relative_time and "ago" in relative_time:
             target_date = today - datetime.timedelta(weeks=1)

        return target_date.strftime("%A, %B %d, %Y")
    except Exception:
        return f"Error calculating date. Today is {today.strftime('%A, %B %d, %Y')}."
   

In [24]:
today_str = datetime.date.today().strftime("%A, %B %d, %Y")

system_prompt = f"""You are the 'Incident Documentation System'. 
Your SOLE purpose is to administratively record details of PAST events.

- Today's Date is: {today_str}


*** INTERVIEW PROTOCOL ***

1.  **Phase 1: Identity**
    -   GOAL: Get First Name, Last Name
    -   ACTION: Ask for user first name and last name
    -   CHECK: If partial info is given, ask specifically for the missing part.
    -   RULE:  Explain why the information is needed if user refuses to comply

2.  **Phase 2: Address**
    -   GOAL: Get the user address
    -   ACTION: Ask for user  physical address, city, and state.
    -   CHECK: If partial info is given, ask specifically for the missing part.
    -   RULE:  Explain why the information is needed if user refuses to comply

3.  **Phase 3: Time & Place**
    -   GOAL: Get Date, Time, and Location.
    -   ACTION: Ask "When and where did this event occur?"
    -   **TOOL USAGE**: If the user says "2 days ago", "yesterday", etc., you MUST call the `calculate_date` tool.
    -   CONFIRM: Once the tool gives you the date, say: "To confirm, that was on [Date] at [Location]. Is that correct?"
 

4.  **Phase 4: Narrative**
    -   GOAL: Get the story.
    -   ACTION: Ask "Please describe the event in detail."
    -   RULE: You must get the WHAT, WHERE, WHY, HOW, MOTIVE, and as many details as possible
    -   CONFIRM: Confirm all the information before jumping to the next phase

5.  **Phase 5: Suspects & Vehicles**
    -   ACTION: Ask if there are descriptions of suspects or vehicles.

6.  **Phase 6: Final Verification**
    -   ACTION: Output a summary of collected data.
    -   ACTION: Ask "Is this accurate? (Yes/No)"

***Output RULES***
 - Do not display the steps, data in json format. Keep the output as natural as possible

*** TOOL RULES ***
-   Only call `submit_incident_report` if the user says "Yes" in Phase 6.
"""

In [25]:
model = ChatOpenAI(model="gpt-4o", temperature=1)

checkPointer = InMemorySaver()

agent = create_agent(
    model,
    tools=[submit_incident_report,calculate_date],
    system_prompt=system_prompt,
    checkpointer=checkPointer
)

In [26]:
config = {"configurable": {"thread_id": str(uuid.uuid4())}}

In [27]:
while True:
    try:
        user_input = input("\nReportee: ")
    except (EOFError, KeyboardInterrupt):
        print("\nExiting...")
        break

    if user_input.lower() in ["exit", "done", "quit"]:
        break

    if not user_input.strip():
        continue

    for step in agent.stream(
        {"messages": [{"role": "user", "content": user_input}]},
        config=config,
        stream_mode="values"
    ):
        last_message = step["messages"][-1]

        if last_message.type == "ai" and last_message.content:
            print(f"Agent: {last_message.content}")

    
        if last_message.type == "tool":
            print(f"\n[System Validator]: {last_message.content}")
            
        



Agent: I can help you with that. Let's start with your identity. Could you please provide me with your first and last name?
Agent: Thank you, Joe. Could you please provide your last name?
Agent: I understand that providing your full name may feel uncomfortable, but it's necessary for the official documentation of the report. Your information will be handled with care. Can you please provide your last name?
Agent: Thank you, Joe Doe. Now let's move on to your address. Could you please provide your physical address, including the city and state?
Agent: Thanks for letting me know that you're in San Francisco. Could you please provide the rest of your address, including the street name and state, to complete the information?
Agent: The full address is necessary for record-keeping and to ensure that any follow-up or correspondence can reach you effectively. It also helps in determining the jurisdiction and area relevant to the incident report. Could you please provide your full address?
Age